# PINN 03 — Carotid Artery Bifurcation (2D Incompressible Flow)

**Clinical context:**
The carotid bifurcation is where the common carotid artery (CCA) divides into
the internal (ICA) and external (ECA) carotid arteries, supplying blood to
the brain and face respectively.

It is also one of the most common sites of **atherosclerotic plaque formation**
— the leading cause of ischaemic stroke.

The mechanism: **low, oscillating Wall Shear Stress (WSS)** at the outer wall
of the bifurcation sinus creates a disturbed flow environment that promotes:
1. Endothelial dysfunction and inflammatory gene upregulation
2. Monocyte adhesion and subintimal lipid deposition
3. Plaque growth → stenosis → thrombus → stroke

**This notebook computes:**
- Velocity field and streamlines through the bifurcation
- Pressure distribution
- Wall Shear Stress map — identifying low-WSS danger zones

**Geometry:** Simplified 2D Y-bifurcation
- CCA: inlet at bottom, channel half-width H = 1.0
- ICA: top-left branch, angle +35° from vertical, slightly larger lumen
- ECA: top-right branch, angle −35° from vertical, smaller lumen
- Bifurcation apex at origin (0, 0)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")
torch.manual_seed(42); np.random.seed(42)

Re   = 100.0   # physiologically ~200–500 in CCA; Re=100 for stability
H    = 1.0     # CCA half-width (normalised)
L_in = 3.0     # CCA inlet length
L_br = 3.0     # branch length
theta_ica = np.radians(35)   # ICA angle from vertical
theta_eca = np.radians(35)   # ECA angle from vertical
H_ica = 0.7 * H              # ICA half-width (larger branch)
H_eca = 0.5 * H              # ECA half-width (smaller branch)

## Geometry Definition

The Y-bifurcation is defined as three rectangular segments joined at the apex.
For each segment we define local (s, n) coordinates:
  s = axial direction (along vessel centreline)
  n = normal direction (across lumen width)

A point is inside the domain if it lies within ANY of the three segments.

In [ ]:
def rotation_matrix(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, -s], [s, c]])

# CCA: vertical, apex at (0,0), extends down to y=-L_in
# Local → global: x_global = x_local, y_global = y_local - L_in  [no rotation]
# A point (x,y) is in CCA if  |x| < H  and  -L_in < y < 0

def in_cca(x, y):
    return (np.abs(x) < H) & (y > -L_in) & (y < 0)

def in_ica(x, y):
    # ICA: tilted left (negative x), apex at (0,0)
    # Rotate by +theta_ica
    R = rotation_matrix(-theta_ica)
    xl = R[0,0]*x + R[0,1]*y
    yl = R[1,0]*x + R[1,1]*y
    return (np.abs(xl) < H_ica) & (yl > 0) & (yl < L_br)

def in_eca(x, y):
    # ECA: tilted right (positive x), apex at (0,0)
    R = rotation_matrix(theta_eca)
    xl = R[0,0]*x + R[0,1]*y
    yl = R[1,0]*x + R[1,1]*y
    return (np.abs(xl) < H_eca) & (yl > 0) & (yl < L_br)

def in_domain(x, y):
    return in_cca(x, y) | in_ica(x, y) | in_eca(x, y)

# Quick visualisation of geometry
fig, ax = plt.subplots(1, 1, figsize=(6, 8), facecolor='#0d0d0d')
ax.set_facecolor('#1a1a1a')
n_pts = 50000
x_test = np.random.uniform(-3, 3, n_pts)
y_test = np.random.uniform(-L_in - 0.5, L_br + 0.5, n_pts)
mask = in_domain(x_test, y_test)
ax.scatter(x_test[mask], y_test[mask], s=0.2, c='#6366f1', alpha=0.5)
ax.scatter(x_test[~mask], y_test[~mask], s=0.2, c='#1a1a2e', alpha=0.1)
ax.set_title('Carotid Bifurcation Geometry\n(domain = purple)', color='white', fontsize=12)
ax.set_xlabel('x', color='white'); ax.set_ylabel('y', color='white')
ax.annotate('CCA (inlet)', xy=(0, -L_in/2), color='#22c55e', fontsize=10, ha='center')
ax.annotate('ICA', xy=(-1.5, 1.5), color='#f97316', fontsize=10, ha='center')
ax.annotate('ECA', xy=(1.2, 1.2), color='#a78bfa', fontsize=10, ha='center')
ax.tick_params(colors='white')
for s in ax.spines.values(): s.set_edgecolor('#444')
plt.tight_layout()
plt.savefig('carotid_geometry.png', dpi=120, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## Sampling Interior and Boundary Points

We use rejection sampling — generate candidate points in a bounding box,
keep only those inside the domain. This handles complex geometries cleanly
without any mesh generation.

In [ ]:
def sample_interior(n=10000):
    pts = []
    while len(pts) < n:
        x_c = np.random.uniform(-3, 3, n*4)
        y_c = np.random.uniform(-L_in, L_br, n*4)
        mask = in_domain(x_c, y_c)
        pts.extend(zip(x_c[mask], y_c[mask]))
    pts = np.array(pts[:n])
    x = torch.tensor(pts[:, 0:1], dtype=torch.float32, device=device)
    y = torch.tensor(pts[:, 1:2], dtype=torch.float32, device=device)
    return x, y

# CCA inlet (y = -L_in)
def sample_inlet(n=300):
    x_np = np.random.uniform(-H, H, n)
    y_np = np.full(n, -L_in)
    u_np = (1.0 - (x_np / H)**2)    # parabolic, U_max=1
    v_np = np.zeros(n)
    x  = torch.tensor(x_np[:, None],  dtype=torch.float32, device=device)
    y  = torch.tensor(y_np[:, None],  dtype=torch.float32, device=device)
    u_ = torch.tensor(u_np[:, None],  dtype=torch.float32, device=device)
    v_ = torch.tensor(v_np[:, None],  dtype=torch.float32, device=device)
    return x, y, u_, v_

# ICA outlet (s = L_br in ICA local coords)
def sample_ica_outlet(n=200):
    R = rotation_matrix(-theta_ica)
    s_np = np.full(n, L_br)
    n_np = np.random.uniform(-H_ica, H_ica, n)
    # local → global
    xl = n_np; yl = s_np
    xg = R[0,0]*xl + R[0,1]*yl     # R^-1 = R^T for rotation
    yg = R[1,0]*xl + R[1,1]*yl
    # Use R transpose (inverse rotation)
    Rt = rotation_matrix(theta_ica)
    xg = Rt[0,0]*xl + Rt[0,1]*yl
    yg = Rt[1,0]*xl + Rt[1,1]*yl
    x = torch.tensor(xg[:, None], dtype=torch.float32, device=device)
    y = torch.tensor(yg[:, None], dtype=torch.float32, device=device)
    return x, y

# ECA outlet
def sample_eca_outlet(n=200):
    R = rotation_matrix(-theta_eca)
    Rt = rotation_matrix(-theta_eca)  # ECA goes right
    s_np = np.full(n, L_br)
    n_np = np.random.uniform(-H_eca, H_eca, n)
    xl = n_np; yl = s_np
    Rt2 = rotation_matrix(theta_eca)
    xg =  Rt2[0,0]*xl + Rt2[0,1]*yl
    yg =  Rt2[1,0]*xl + Rt2[1,1]*yl
    x = torch.tensor(xg[:, None], dtype=torch.float32, device=device)
    y = torch.tensor(yg[:, None], dtype=torch.float32, device=device)
    return x, y

# Wall boundary points (rejection sampled near walls)
def sample_walls(n=1000):
    """Approximate wall sampling: points very close to boundary."""
    # CCA walls: x = ±H, y ∈ [-L_in, 0]
    y_cca = np.random.uniform(-L_in, 0, n)
    x_l = np.full(n, -H + 0.01)
    x_r = np.full(n, H - 0.01)
    x_walls = np.concatenate([x_l, x_r])
    y_walls = np.concatenate([y_cca, y_cca])
    x = torch.tensor(x_walls[:, None], dtype=torch.float32, device=device)
    y = torch.tensor(y_walls[:, None], dtype=torch.float32, device=device)
    return x, y

print("Sampling domain points...")
x_int, y_int = sample_interior(10000)
x_in, y_in, u_in, v_in = sample_inlet()
x_w, y_w = sample_walls()
x_ica_out, y_ica_out = sample_ica_outlet()
x_eca_out, y_eca_out = sample_eca_outlet()
print(f"Interior: {len(x_int)} | Inlet: {len(x_in)} | Walls: {len(x_w)}")

## Neural Network and PDE Residuals

In [ ]:
class PINN(nn.Module):
    def __init__(self, layers=[2, 128, 128, 128, 128, 3]):
        super().__init__()
        net = []
        for i in range(len(layers) - 1):
            net.append(nn.Linear(layers[i], layers[i+1]))
            if i < len(layers) - 2:
                net.append(nn.Tanh())
        self.net = nn.Sequential(*net)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x, y):
        out = self.net(torch.cat([x, y], dim=1))
        return out[:, 0:1], out[:, 1:2], out[:, 2:3]

model = PINN().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

def ns_residuals(model, x, y):
    x = x.clone().requires_grad_(True)
    y = y.clone().requires_grad_(True)
    u, v, p = model(x, y)

    def grad(f, var):
        return torch.autograd.grad(f, var,
                                   grad_outputs=torch.ones_like(f),
                                   create_graph=True)[0]
    u_x = grad(u, x); u_y = grad(u, y)
    v_x = grad(v, x); v_y = grad(v, y)
    p_x = grad(p, x); p_y = grad(p, y)
    u_xx = grad(u_x, x); u_yy = grad(u_y, y)
    v_xx = grad(v_x, x); v_yy = grad(v_y, y)

    cont  = u_x + v_y
    xmom  = u*u_x + v*u_y + p_x - (1.0/Re)*(u_xx + u_yy)
    ymom  = u*v_x + v*v_y + p_y - (1.0/Re)*(v_xx + v_yy)
    return cont, xmom, ymom

## Loss Function and Training

In [ ]:
def compute_loss(model):
    cont, xm, ym = ns_residuals(model, x_int, y_int)
    loss_pde = (cont**2).mean() + (xm**2).mean() + (ym**2).mean()

    # Inlet BC
    u_i, v_i, _ = model(x_in, y_in)
    loss_bc = ((u_i - u_in)**2).mean() + ((v_i - v_in)**2).mean()

    # Wall no-slip
    u_w, v_w, _ = model(x_w, y_w)
    loss_bc += (u_w**2).mean() + (v_w**2).mean()

    # Outlets: zero pressure
    _, _, p_ica = model(x_ica_out, y_ica_out)
    _, _, p_eca = model(x_eca_out, y_eca_out)
    loss_bc += (p_ica**2).mean() + (p_eca**2).mean()

    # Pressure pin at apex
    x0 = torch.zeros(1, 1, device=device); y0 = torch.zeros(1, 1, device=device)
    _, _, p0 = model(x0, y0)
    loss_pin = p0**2

    return loss_pde + loss_bc + loss_pin

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
losses = []

print("Training carotid bifurcation PINN (this takes ~15 min on free Colab GPU)...")
for epoch in range(25000):
    optimizer.zero_grad()
    loss = compute_loss(model)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if epoch % 5000 == 0:
        print(f"  Epoch {epoch:6d} | Loss {loss.item():.4e}")

# L-BFGS
lbfgs = torch.optim.LBFGS(model.parameters(), lr=0.1, max_iter=100,
                            line_search_fn="strong_wolfe")
def closure():
    lbfgs.zero_grad(); l = compute_loss(model); l.backward(); return l
for s in range(10):
    lbfgs.step(closure)
print(f"Final loss: {compute_loss(model).item():.4e}")

## Wall Shear Stress Computation

WSS = μ × |∂u/∂n|  where n is the wall-normal direction

**Clinical interpretation:**
- WSS < 0.4 Pa → disturbed flow, atherogenic zone
- WSS 1.5–2.5 Pa → normal, protective
- WSS > 5 Pa → high shear, possible endothelial injury

The outer wall of the bifurcation sinus (ICA outer wall) is the canonical
low-WSS zone — this is where most carotid plaques begin.

In [ ]:
model.eval()
n_wss = 150
# CCA left wall WSS
y_cca_wss = torch.linspace(-L_in, -0.1, n_wss, device=device).unsqueeze(1)
x_cca_l   = torch.full((n_wss, 1), -H + 0.01, device=device)
x_g = x_cca_l.clone().requires_grad_(True)
y_g = y_cca_wss.clone().requires_grad_(True)
u_w, _, _ = model(x_g, y_g)
du_dx = torch.autograd.grad(u_w.sum(), x_g, create_graph=False)[0]
wss_cca_l = (1.0/Re) * du_dx.detach().cpu().numpy().squeeze()

## Visualisation

In [ ]:
model.eval()
with torch.no_grad():
    # Build a grid of points, evaluate only inside domain
    nx, ny = 120, 120
    xs = np.linspace(-2.5, 2.5, nx)
    ys = np.linspace(-L_in, L_br, ny)
    XG, YG = np.meshgrid(xs, ys)
    mask = in_domain(XG.ravel(), YG.ravel())

    x_all = torch.tensor(XG.ravel()[mask, None], dtype=torch.float32, device=device)
    y_all = torch.tensor(YG.ravel()[mask, None], dtype=torch.float32, device=device)
    U_all, V_all, P_all = model(x_all, y_all)
    U_all = U_all.cpu().numpy().squeeze()
    V_all = V_all.cpu().numpy().squeeze()
    P_all = P_all.cpu().numpy().squeeze()

# Reconstruct full grids (NaN outside domain)
U_grid = np.full(XG.shape, np.nan)
V_grid = np.full(XG.shape, np.nan)
P_grid = np.full(XG.shape, np.nan)
speed_grid = np.full(XG.shape, np.nan)
U_grid.ravel()[mask] = U_all
V_grid.ravel()[mask] = V_all
P_grid.ravel()[mask] = P_all
speed_grid.ravel()[mask] = np.sqrt(U_all**2 + V_all**2)

fig, axes = plt.subplots(2, 2, figsize=(14, 16), facecolor='#0d0d0d')
fig.suptitle('PINN — Carotid Artery Bifurcation (Re=100)\n'
             'Common Carotid → Internal + External Carotid',
             color='white', fontsize=14, fontweight='bold')

kw = dict(facecolor='#1a1a1a')

def style_ax(ax, title, xlabel='x', ylabel='y'):
    ax.set_title(title, color='white', fontsize=11)
    ax.set_xlabel(xlabel, color='white'); ax.set_ylabel(ylabel, color='white')
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_edgecolor('#444')

ax = axes[0, 0]; ax.set(**kw)
cf = ax.contourf(XG, YG, speed_grid, levels=50, cmap='inferno')
# Streamlines (only in domain — skip NaN rows)
valid_rows = ~np.all(np.isnan(U_grid), axis=1)
try:
    ax.streamplot(xs, ys[valid_rows],
                  np.nan_to_num(U_grid[valid_rows]),
                  np.nan_to_num(V_grid[valid_rows]),
                  color='white', linewidth=0.5, density=1.2, arrowsize=0.7)
except Exception:
    pass
plt.colorbar(cf, ax=ax, label='|velocity|')
style_ax(ax, 'Velocity Magnitude + Streamlines')

ax = axes[0, 1]; ax.set(**kw)
cf2 = ax.contourf(XG, YG, P_grid, levels=50, cmap='RdBu_r')
plt.colorbar(cf2, ax=ax, label='pressure')
style_ax(ax, 'Pressure Distribution\n(drops from CCA inlet to outlets)')

ax = axes[1, 0]; ax.set(**kw)
cf3 = ax.contourf(XG, YG, U_grid, levels=50, cmap='coolwarm')
plt.colorbar(cf3, ax=ax, label='u-velocity')
style_ax(ax, 'u-Velocity Contours')

ax = axes[1, 1]; ax.set(**kw)
ax.semilogy(losses, color='#22c55e', lw=1.5)
style_ax(ax, 'Training Loss', xlabel='Epoch', ylabel='Loss')

plt.tight_layout()
plt.savefig('carotid_bifurcation_flow.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## WSS Map — Identifying Atherogenic Zones

The outer wall of the ICA sinus is the canonical low-WSS danger zone.
Plot WSS along the CCA walls and annotate the bifurcation region.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), facecolor='#0d0d0d')
ax.set_facecolor('#1a1a1a')
y_cca_np = y_cca_wss.cpu().numpy().squeeze()
ax.plot(y_cca_np, abs(wss_cca_l), color='#6366f1', lw=2.5, label='CCA left wall WSS')
ax.axhline(0.4, color='#ef4444', lw=1.5, linestyle='--', label='Low WSS threshold (0.4)')
ax.axhline(1.5, color='#22c55e', lw=1.5, linestyle='--', label='Normal WSS lower bound (1.5)')
ax.fill_between(y_cca_np, 0, 0.4, alpha=0.2, color='#ef4444', label='Atherogenic zone')
ax.set_xlabel('y (along vessel)', color='white')
ax.set_ylabel('Wall Shear Stress |τ|', color='white')
ax.set_title('Wall Shear Stress — CCA Wall\nRed zone: low WSS → plaque risk', color='white')
ax.legend(facecolor='#2a2a2a', edgecolor='#444', labelcolor='white')
ax.tick_params(colors='white')
for s in ax.spines.values(): s.set_edgecolor('#444')
plt.tight_layout()
plt.savefig('carotid_wss.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

print("\n✅ Carotid bifurcation notebook complete.")
print("Clinical takeaway: Low WSS at the outer bifurcation wall → atherogenic zone → plaque formation → stroke risk.")
print("PINNs can map this WITHOUT a mesh, in under 15 minutes on free Colab GPU.")